**Install Required Packages**

In [1]:
!pip install nltk spacy matplotlib numpy pandas scikit-learn
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
print("Packages installed successfully!\n")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Packages installed successfully!



# **Task 1 - Minimum Edit Distance**

In [2]:
def minimum_edit_distance(str1, str2, sub_cost=2):
    n = len(str1)
    m = len(str2)

    dp = [[0 for _ in range(m + 1)] for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if str1[i - 1] == str2[j - 1]:
                substitution_cost = 0
            else:
                substitution_cost = sub_cost

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + substitution_cost
            )

    return dp[n][m]

text1 = input("Enter first text: ")
text2 = input("Enter second text: ")

distance = minimum_edit_distance(text1, text2)
print(f"Minimum Edit Distance between '{text1}' and '{text2}': {distance}")
print()

Enter first text: INTENTION
Enter second text: EXECUTION
Minimum Edit Distance between 'INTENTION' and 'EXECUTION': 8



# **Task 2: Fake News Detection**

**Mount Google Drive and Load Dataset**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df_fake = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Fake_news_detection/News_dataset/Fake.csv")
df_true = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Fake_news_detection/News_dataset/True.csv")

print("Datasets loaded successfully!")
print(f"Fake news samples: {len(df_fake)}")
print(f"True news samples: {len(df_true)}\n")

Mounted at /content/drive
Datasets loaded successfully!
Fake news samples: 23481
True news samples: 21417



**Prepare Balanced Dataset**

In [4]:
df_fake_sample = df_fake[['title', 'text']].head(100)
df_fake_sample = df_fake_sample.rename(columns={'text': 'content'})
df_fake_sample['label'] = 1

df_true_sample = df_true[['title', 'text']].head(100)
df_true_sample = df_true_sample.rename(columns={'text': 'content'})
df_true_sample['label'] = 0

df = pd.concat([df_fake_sample, df_true_sample], ignore_index=True)

print("Balanced dataset created")
print(f"Total samples: {len(df)}")
print(f"Fake news (label=1): {len(df[df['label'] == 1])}")
print(f"True news (label=0): {len(df[df['label'] == 0])}")
print("\nFirst 10 rows:")
print(df[['title', 'label']].head(10))
print()

Balanced dataset created
Total samples: 200
Fake news (label=1): 100
True news (label=0): 100

First 10 rows:
                                               title  label
0   Donald Trump Sends Out Embarrassing New Year’...      1
1   Drunk Bragging Trump Staffer Started Russian ...      1
2   Sheriff David Clarke Becomes An Internet Joke...      1
3   Trump Is So Obsessed He Even Has Obama’s Name...      1
4   Pope Francis Just Called Out Donald Trump Dur...      1
5   Racist Alabama Cops Brutalize Black Boy While...      1
6   Fresh Off The Golf Course, Trump Lashes Out A...      1
7   Trump Said Some INSANELY Racist Stuff Inside ...      1
8   Former CIA Director Slams Trump Over UN Bully...      1
9   WATCH: Brand-New Pro-Trump Ad Features So Muc...      1



**Tokenization**

In [5]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')

df['tokens'] = df['content'].apply(word_tokenize)

print("Tokenization completed")
print(f"Sample tokens from first row: {df['tokens'].iloc[0][:10]}")
print(f"Number of tokens in first sample: {len(df['tokens'].iloc[0])}")
print()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Tokenization completed
Sample tokens from first row: ['Donald', 'Trump', 'just', 'couldn', 't', 'wish', 'all', 'Americans', 'a', 'Happy']
Number of tokens in first sample: 599



**Case Folding**

In [6]:
df['case_folded'] = df['content'].str.lower()
df['tokens'] = df['case_folded'].apply(word_tokenize)

print("Case folding completed")
print(f"Sample after case folding: {df['tokens'].iloc[0][:10]}")
print()

Case folding completed
Sample after case folding: ['donald', 'trump', 'just', 'couldn', 't', 'wish', 'all', 'americans', 'a', 'happy']



**Punctuation Removal**

In [7]:
punct = set('''!()-[]{};:'"\\,<>./?@#$%^&*_~''')
df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in punct])

print("Punctuation removal completed")
print(f"Sample after punctuation removal: {df['tokens'].iloc[0][:10]}")
print(f"Number of tokens after punctuation removal: {len(df['tokens'].iloc[0])}")
print()

Punctuation removal completed
Sample after punctuation removal: ['donald', 'trump', 'just', 'couldn', 't', 'wish', 'all', 'americans', 'a', 'happy']
Number of tokens after punctuation removal: 492



**Stop Words Removal**

In [8]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])

print("Stop words removal completed")
print(f"Sample after stop words removal: {df['tokens'].iloc[0][:10]}")
print(f"Number of tokens after stop words removal: {len(df['tokens'].iloc[0])}")
print()

Stop words removal completed
Sample after stop words removal: ['donald', 'trump', 'wish', 'americans', 'happy', 'new', 'year', 'leave', 'instead', 'give']
Number of tokens after stop words removal: 287



**Stemming**

In [9]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
df['stemmed'] = df['tokens'].apply(lambda x: [stemmer.stem(word) for word in x])

print("Stemming completed")
print(f"Original tokens: {df['tokens'].iloc[0][:8]}")
print(f"Stemmed tokens: {df['stemmed'].iloc[0][:8]}")
print()

Stemming completed
Original tokens: ['donald', 'trump', 'wish', 'americans', 'happy', 'new', 'year', 'leave']
Stemmed tokens: ['donald', 'trump', 'wish', 'american', 'happi', 'new', 'year', 'leav']



**Lemmatization**

In [10]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
df['lemmatized'] = df['tokens'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

print("Lemmatization completed")
print(f"Original tokens: {df['tokens'].iloc[0][:8]}")
print(f"Lemmatized tokens: {df['lemmatized'].iloc[0][:8]}")
print()

Lemmatization completed
Original tokens: ['donald', 'trump', 'wish', 'americans', 'happy', 'new', 'year', 'leave']
Lemmatized tokens: ['donald', 'trump', 'wish', 'american', 'happy', 'new', 'year', 'leave']



**Synonym Substitution**

In [11]:
from nltk.corpus import wordnet

def get_synonym(word):
    synonyms = wordnet.synsets(word)
    if synonyms:
        lemmas = synonyms[0].lemmas()
        if lemmas and lemmas[0].name() != word:
            return lemmas[0].name()
    return word

df['synonym_sub'] = df['tokens'].apply(lambda x: [get_synonym(word) for word in x[:5]] + x[5:])

print("Synonym substitution completed")
print(f"Original tokens: {df['tokens'].iloc[0][:5]}")
print(f"After synonym substitution: {df['synonym_sub'].iloc[0][:5]}")
print()

Synonym substitution completed
Original tokens: ['donald', 'trump', 'wish', 'americans', 'happy']
After synonym substitution: ['donald', 'trump', 'wish', 'American', 'happy']



**Prepare Final Text for Modeling**

In [12]:
df['final_text'] = df['lemmatized'].apply(lambda x: ' '.join(x))

print("Final text preparation completed")
print(f"Sample final text (first 100 chars): {df['final_text'].iloc[0][:100]}...")
print(f"Final dataset shape: {df[['final_text', 'label']].shape}")
print()

Final text preparation completed
Sample final text (first 100 chars): donald trump wish american happy new year leave instead give shout enemy hater dishonest fake news m...
Final dataset shape: (200, 2)



**Vector Semantics - TF-IDF Vectorization**

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

X = df['final_text']
y = df['label']

X_tfidf = tfidf.fit_transform(X)

print("TF-IDF vectorization completed")
print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Sample feature names: {list(tfidf.vocabulary_.keys())[:10]}")
print()

TF-IDF vectorization completed
TF-IDF matrix shape: (200, 5000)
Vocabulary size: 5000
Sample feature names: ['donald', 'trump', 'wish', 'american', 'happy', 'new', 'year', 'leave', 'instead', 'shout']



**Train-Test Split**

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Training features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"Class distribution in test set:")
print(f"  True news (0): {sum(y_test == 0)}")
print(f"  Fake news (1): {sum(y_test == 1)}")
print()

Training set size: 160
Test set size: 40
Training features shape: (160, 5000)
Test features shape: (40, 5000)
Class distribution in test set:
  True news (0): 20
  Fake news (1): 20



**Multinomial Naive Bayes Model Training**

In [15]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

print("Multinomial Naive Bayes model trained successfully!")
print(f"Model parameters: alpha={model.alpha}, fit_prior={model.fit_prior}")
print()

Multinomial Naive Bayes model trained successfully!
Model parameters: alpha=1.0, fit_prior=True



**Model Predictions**

In [16]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("Predictions completed")
print(f"First 10 predictions: {y_pred[:10]}")
print(f"First 10 actual labels: {y_test.values[:10]}")
print(f"Prediction probabilities for first sample: {y_pred_proba[0]}")
print(f"Confidence for first prediction: {max(y_pred_proba[0])*100:.2f}%")
print()

Predictions completed
First 10 predictions: [0 1 1 1 0 1 0 0 1 0]
First 10 actual labels: [0 1 1 1 0 1 0 1 1 0]
Prediction probabilities for first sample: [0.98186514 0.01813486]
Confidence for first prediction: 98.19%



**Model Evaluation**


In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("COMPREHENSIVE MODEL EVALUATION")
print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1 Score:  {f1 * 100:.2f}%")

COMPREHENSIVE MODEL EVALUATION
Accuracy:  95.00%
Precision: 100.00%
Recall:    90.00%
F1 Score:  94.74%


**Fake News Prediction Function**

In [19]:
def predict_news(text):
    text_lower = text.lower()
    tokens = word_tokenize(text_lower)
    tokens = [word for word in tokens if word not in punct]
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    processed_text = ' '.join(tokens)

    text_tfidf = tfidf.transform([processed_text])

    prediction = model.predict(text_tfidf)[0]
    probability = model.predict_proba(text_tfidf)[0]

    if prediction == 1:
        return "FAKE NEWS", probability[1] * 100
    else:
        return "TRUE NEWS", probability[0] * 100

test_texts = [
    "Donald Trump Sends Out Embarrassing New Year’s Eve Message To CNN Reporter Jeff Zeleny",
     "Aliens landed in Washington DC and took control of the government yesterday",
    "The president announced new economic policies to boost the national economy",
    "Celebrity claims drinking bleach cures all diseases without any side effects"
]

print("TESTING FAKE NEWS DETECTOR")

for i, text in enumerate(test_texts, 1):
    prediction, confidence = predict_news(text)
    print(f"\nTest {i}:")
    print(f"Text: {text[:60]}...")
    print(f"Prediction: {prediction}")

TESTING FAKE NEWS DETECTOR

Test 1:
Text: Donald Trump Sends Out Embarrassing New Year’s Eve Message T...
Prediction: FAKE NEWS

Test 2:
Text: Aliens landed in Washington DC and took control of the gover...
Prediction: TRUE NEWS

Test 3:
Text: The president announced new economic policies to boost the n...
Prediction: TRUE NEWS

Test 4:
Text: Celebrity claims drinking bleach cures all diseases without ...
Prediction: FAKE NEWS
